# Assignment 20: Building & Deploying a Recommendation System
**Name:** Suman Rajak

**Dataset:** TMDB 5000 Movie Dataset (Kaggle)
**Link:** https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata
File used: `tmdb_5000_movies.csv` (4,803 movies)

- Item identifier: `title` (shown with the release year, because a few titles repeat)
- Text features: `overview`, `tagline`, `genres`, `keywords`

This notebook covers Parts 1–3 (preprocessing, TF-IDF, similarity, recommend function). The Streamlit app is in `app.py`.
Libraries: Python, Pandas, scikit-learn (and Streamlit for the app).

---
# PART 1 - Data Preprocessing

## Task 1: Load & Understand Dataset

In [1]:
import json
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

movies = pd.read_csv("data/tmdb_5000_movies.csv")
print("Shape:", movies.shape)
print("Columns:", list(movies.columns))

Shape: (4803, 20)
Columns: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']


In [2]:
movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 878, ""name"": ""...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id"": 3386, ""name"": ""space war""}, {""id"": 33...",en,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn bet...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289}, {""name"": ""Twentieth Century Fox Film Corporation"", ""id"": 306}, {""na...","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}, {""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}]",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639_1"": ""es"", ""name"": ""Espa\u00f1ol""}]",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""name"": ""Action""}]",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug abuse""}, {""id"": 911, ""name"": ""exotic island""}, {""id"": 1319,...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will T...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""name"": ""Jerry Bruckheimer Films"", ""id"": 130}, {""name"": ""Second Mate Pr...","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}]",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 80, ""name"": ""Crime""}]",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name"": ""based on novel""}, {""id"": 4289, ""name"": ""secret agent""}, {""id"": 966...",en,Spectre,A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles politica...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""name"": ""Danjaq"", ""id"": 10761}, {""name"": ""B24"", ""id"": 69434}]","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}, {""iso_3166_1"": ""US"", ""name"": ""United States of America""}]",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""}, {""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639_1"": ""es"", ""name"": ""...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""name"": ""Crime""}, {""id"": 18, ""name"": ""Drama""}, {""id"": 53, ""name"": ""Thrille...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853, ""name"": ""crime fighter""}, {""id"": 949, ""name"": ""terrorist""}, {""id"": 13...",en,The Dark Knight Rises,"Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's crimes to protect the...",112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""name"": ""Warner Bros."", ""id"": 6194}, {""name"": ""DC Entertainment"", ""id"":...","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}]",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 878, ""name"": ""Science Fiction""}]",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""n

In [3]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   str    
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   str    
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

In [4]:
print("Missing values in useful columns:")
print(movies[["title", "overview", "tagline", "genres", "keywords", "release_date"]].isnull().sum())
print("\nDuplicate titles:", movies["title"].duplicated().sum())
print(movies[movies["title"].duplicated(keep=False)][["title", "release_date"]])

Missing values in useful columns:
title             0
overview          3
tagline         844
genres            0
keywords          0
release_date      1
dtype: int64

Duplicate titles: 3
                title release_date
972          The Host   2013-03-22
1359           Batman   1989-06-23
2877         The Host   2006-07-27
3647  Out of the Blue   1980-05-01
3693  Out of the Blue   2006-10-12
4267           Batman   1966-07-30


In [5]:
# genres and keywords are stored as JSON text
print("genres  :", movies.loc[0, "genres"][:150])
print("keywords:", movies.loc[0, "keywords"][:150])

genres  : [{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]
keywords: [{"id": 1463, "name": "culture clash"}, {"id": 2964, "name": "future"}, {"id": 3386, "name": "space war"}, {"id": 3388, "name": "space colony"}, {"id"


**Text columns used for recommendations:**
- `overview`: short plot summary (the main description)
- `tagline`: one-line slogan (844 movies don't have one)
- `genres`: e.g. Action, Adventure, Science Fiction
- `keywords`: tags like "space war", "alien", "future"

`genres` and `keywords` are JSON lists, so I extract just the names. Three titles appear twice (Batman, The Host, Out of the Blue) for different movies, so I add the year to make each name unique.

## Task 2: Text Preprocessing

In [6]:
def extract_names(json_text):
    """'[{"id": 28, "name": "Science Fiction"}]' -> 'sciencefiction'"""
    items = json.loads(json_text) if isinstance(json_text, str) else []
    # remove spaces inside a name so multi-word tags stay one token
    return " ".join(item["name"].replace(" ", "").lower() for item in items)

movies["genre_names"] = movies["genres"].apply(lambda g: ", ".join(i["name"] for i in json.loads(g)))
movies["genres_text"] = movies["genres"].apply(extract_names)
movies["keywords_text"] = movies["keywords"].apply(extract_names)

# 4. Missing values -> empty string
for col in ["overview", "tagline"]:
    movies[col] = movies[col].fillna("")

# Unique display name: "Title (Year)"
movies["year"] = movies["release_date"].fillna("").str[:4]
movies["display_title"] = movies.apply(lambda r: f"{r['title']} ({r['year']})" if r["year"] else r["title"], axis=1)
print("Unique display titles:", movies["display_title"].is_unique)

movies[["display_title", "genres_text", "keywords_text"]].head()

Unique display titles: True


,display_title,genres_text,keywords_text
0,Avatar (2009),action adventure fantasy sciencefiction,cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi ma...
1,Pirates of the Caribbean: At World's End (2007),adventure fantasy action,ocean drugabuse exoticisland eastindiatradingcompany loveofone'slife traitor shipwreck strongwoman ship alliance cal...
2,Spectre (2015),action adventure crime,spy basedonnovel secretagent sequel mi6 britishsecretservice unitedkingdom
3,The Dark Knight Rises (2012),action crime drama thriller,dccomics crimefighter terrorist secretidentity burglar hostagedrama timebomb gothamcity vigilante cover-up superhero...
4,John Carter (2012),action adventure sciencefiction,basedonnovel mars medallion spacetravel princess alien steampunk martian escape edgarriceburroughs alienrace superhu...


In [7]:
stop_words = ENGLISH_STOP_WORDS   # scikit-learn's built-in English stopword list

def clean(text):
    text = text.lower()                                  # 1. lowercase
    text = re.sub(r"[^a-z\s]", " ", text)                # 2. remove punctuation, numbers, special characters
    words = [w for w in text.split() if w not in stop_words and len(w) > 2]   # 3. remove stopwords (and 1-2 letter leftovers like "nd")
    return " ".join(words)

# combine all text features into one "tags" string, then clean it
movies["combined"] = (movies["overview"] + " " + movies["tagline"] + " " +
                      movies["genres_text"] + " " + movies["keywords_text"])
movies["clean_text"] = movies["combined"].apply(clean)

print("Empty clean_text rows:", (movies["clean_text"] == "").sum())
movies[["display_title", "clean_text"]].head()

Empty clean_text rows: 0


,display_title,clean_text
0,Avatar (2009),century paraplegic marine dispatched moon pandora unique mission torn following orders protecting alien civilization...
1,Pirates of the Caribbean: At World's End (2007),captain barbossa long believed dead come life headed edge earth turner elizabeth swann quite end world adventure beg...
2,Spectre (2015),cryptic message bond past sends trail uncover sinister organization battles political forces secret service alive bo...
3,The Dark Knight Rises (2012),following death district attorney harvey dent batman assumes responsibility dent crimes protect late attorney reputa...
4,John Carter (2012),john carter war weary military captain inexplicably transported mysterious exotic planet barsoom mars reluctantly em...


In [8]:
print("Before:", movies.loc[0, "combined"][:300])
print()
print("After :", movies.loc[0, "clean_text"][:300])

Before: In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Enter the World of Pandora. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetr

After : century paraplegic marine dispatched moon pandora unique mission torn following orders protecting alien civilization enter world pandora action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine so


---
# PART 2 - Text Vectorization

## Task 3: TF-IDF Vectorization

In [9]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(movies["clean_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape, "(movies x features)")
print("Sample features:", list(tfidf.get_feature_names_out()[1000:1015]))

TF-IDF matrix shape: (4803, 5000) (movies x features)
Sample features: ['creator', 'creators', 'creature', 'creatures', 'credit', 'creepy', 'crew', 'crime', 'crime action', 'crime boss', 'crime comedy', 'crime corruption', 'crime drama', 'crime family', 'crime horror']


- `max_features=5000` keeps the 5,000 most frequent terms. This removes very rare words (typos, names used once) and keeps the matrix small.
- `ngram_range=(1, 2)` adds two-word phrases like "serial killer", "world war" and "high school", which carry more meaning than the single words.

## Task 4: Similarity Computation

In [10]:
similarity = cosine_similarity(tfidf_matrix)

print("Similarity matrix shape:", similarity.shape)
print("Memory used: about", round(similarity.nbytes / 1024**2), "MB")
print("Similarity of a movie with itself:", round(similarity[0, 0], 3))

pd.DataFrame(similarity[:5, :5], index=movies["display_title"][:5],
             columns=movies["display_title"][:5]).round(3)

Similarity matrix shape: (4803, 4803)
Memory used: about 176 MB
Similarity of a movie with itself: 1.0


display_title,Avatar (2009),Pirates of the Caribbean: At World's End (2007),Spectre (2015),The Dark Knight Rises (2012),John Carter (2012)
display_title,,,,,
Avatar (2009),1.000,0.052,0.024,0.020,0.101
Pirates of the Caribbean: At World's End (2007),0.052,1.000,0.019,0.003,0.051
Spectre (2015),0.024,0.019,1.000,0.009,0.037
The Dark Knight Rises (2012),0.020,0.003,0.009,1.000,0.013
John Carter (2012),0.101,0.051,0.037,0.013,1.000


**Why cosine similarity?**
Cosine similarity measures the **angle** between two vectors, not their length. A movie with a long overview has bigger raw TF-IDF values than one with a short overview, but if they talk about the same things, their vectors point in the same direction. The score goes from 0 (no words in common) to 1 (same direction), so it's easy to read and compare. It's also fast on sparse TF-IDF vectors.

The full matrix needs about 180 MB, so I don't save it to a file. In `app.py`, I compute only the one row needed for the selected movie, which keeps the app light enough for Render's free plan.

---
# PART 3 - Recommendation Logic

## Task 5: Recommendation Function

In [11]:
index_of = pd.Series(movies.index, index=movies["display_title"])

def recommend(item_name, top_n=5):
    """Return the top_n movies most similar to item_name."""
    if item_name not in index_of:
        print("Movie not found:", item_name)
        return pd.DataFrame()

    idx = index_of[item_name]                                  # 1. index of the selected movie
    scores = list(enumerate(similarity[idx]))                  # 2. similarity with every movie
    scores = sorted(scores, key=lambda x: x[1], reverse=True)  # 3. sort, highest first
    scores = [s for s in scores if s[0] != idx][:top_n]        #    skip the movie itself

    result = movies.iloc[[i for i, _ in scores]][["display_title", "genre_names", "vote_average"]].copy()
    result["similarity"] = [round(s, 3) for _, s in scores]
    return result.reset_index(drop=True)

In [12]:
print("Because you watched: Avatar (2009)")
recommend("Avatar (2009)")

Because you watched: Avatar (2009)


,display_title,genre_names,vote_average,similarity
0,Lara Croft Tomb Raider: The Cradle of Life (2003),"Action, Adventure, Fantasy, Thriller",5.5,0.226
1,Titan A.E. (2000),"Animation, Action, Science Fiction, Family, Adventure",6.3,0.183
2,Battle: Los Angeles (2011),"Action, Science Fiction",5.5,0.179
3,Falcon Rising (2014),"Adventure, Action",5.5,0.178
4,Apollo 18 (2011),"Horror, Thriller, Science Fiction",5.0,0.170


In [13]:
print("Because you watched: The Dark Knight (2008)")
recommend("The Dark Knight (2008)")

Because you watched: The Dark Knight (2008)


,display_title,genre_names,vote_average,similarity
0,The Dark Knight Rises (2012),"Action, Crime, Drama, Thriller",7.6,0.445
1,Batman Returns (1992),"Action, Fantasy",6.6,0.378
2,Batman Begins (2005),"Action, Crime, Drama",7.5,0.333
3,Batman Forever (1995),"Action, Crime, Fantasy",5.2,0.333
4,"Batman: The Dark Knight Returns, Part 2 (2013)","Action, Animation",7.9,0.301


In [14]:
print("Because you watched: Toy Story (1995)")
recommend("Toy Story (1995)")

Because you watched: Toy Story (1995)


,display_title,genre_names,vote_average,similarity
0,Toy Story 2 (1999),"Animation, Comedy, Family",7.3,0.490
1,Toy Story 3 (2010),"Animation, Family, Comedy",7.6,0.457
2,The 40 Year Old Virgin (2005),"Comedy, Romance",6.2,0.342
3,Man on the Moon (1999),"Comedy, Drama, Romance",6.9,0.265
4,Factory Girl (2006),Drama,6.2,0.230


In [15]:
print("Because you watched: Titanic (1997)")
recommend("Titanic (1997)")

Because you watched: Titanic (1997)


,display_title,genre_names,vote_average,similarity
0,Raise the Titanic (1980),"Action, Drama, Thriller",5.2,0.359
1,Ghost Ship (2002),"Horror, Mystery, Thriller",5.3,0.187
2,Poseidon (2006),"Adventure, Action, Drama, Thriller",5.5,0.185
3,Captain Phillips (2013),"Action, Drama, Thriller",7.6,0.172
4,The Black Hole (1979),"Adventure, Family, Science Fiction, Action",6.1,0.170


In [16]:
recommend("Some Movie That Does Not Exist")

Movie not found: Some Movie That Does Not Exist


""


**What I noticed in the results:**
- **The Dark Knight** works best: all 5 recommendations are Batman films, because they share words like "batman", "gotham" and "joker".
- **Avatar** gets action and science-fiction adventures (Titan A.E., Battle: Los Angeles, Apollo 18). These share words like "alien", "mission", "space" and "sciencefiction".
- **Titanic** gets ship and sea-disaster films (Raise the Titanic, Poseidon, Ghost Ship). The text about the ship and the ocean outweighs the love story.
- **Toy Story** first gets its own sequels, which is good. But then come The 40 Year Old Virgin and Man on the Moon, only because all three overviews mention a character named **"Andy"**.

This shows the main limitation of content-based filtering with TF-IDF: it matches **words**, not meaning. A character name can make two unrelated movies look similar. All results come only from the text (overview, tagline, genres and keywords), with no user ratings.

In [17]:
# Save the cleaned data for the Streamlit app
movies[["display_title", "title", "year", "genre_names", "overview", "vote_average", "clean_text"]].to_csv(
    "data/movies_clean.csv", index=False)
print("Saved data/movies_clean.csv")

Saved data/movies_clean.csv
